# Competitive CeNN — theoretical feasibility before SmolLM2-360M replacement

**Goal:** Test a new standalone cellular-memory candidate *before* replacing another pretrained attention layer. First run CPU mathematical and synthetic learning tests; then on a Tesla T4 / other CUDA GPU inspect the **original SmolLM2-360M-Instruct layer-0 attention distributions**. No pretrained layer is replaced by this notebook.

**Critical scope:** content-dependent cell routing is functionally **attention-like** although the candidate has no Transformer attention block, no QKV attention matrix, no SDPA and no DeltaNet/LSTM. Novelty and speedups are **not established**.

**Experiment design:** Memory scales with cell count; local nearest-neighbor winners travel up a binary tree. If native attention is diffuse, hard selection cannot preserve it. This notebook checks that hypothesis rather than publishing another failed checkpoint.

In [1]:
#@title 1. Prepare environment
import os, sys, subprocess, json, pathlib, torch
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():print('Device:',torch.cuda.get_device_name())
# Use pinned Transformers to preserve the Llama attention/output_attentions API.
subprocess.check_call([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pytest'])
root=pathlib.Path('/content/TinyCeNN-LM') if pathlib.Path('/content').exists() else pathlib.Path.cwd()/'TinyCeNN-LM'
BRANCH='research/competitive-cenn-feasibility'
if not root.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(root)])
subprocess.check_call([sys.executable,'-m','pip','-q','install','-e',str(root),'--no-deps'])
os.chdir(root)
print('Repository:',root)

PyTorch: 2.11.0+cu130 CUDA: True
Device: Tesla T4
Repository: /content/TinyCeNN-LM


In [2]:
#@title 2. Run causal and gradient tests (CPU)
subprocess.check_call([sys.executable,'-m','pytest','-q','tests/test_competitive_cenn.py'])

0

In [3]:
#@title 3. New cellular layer vs peaked, content-selective teacher
import importlib.util
from tinycenn_lm.competitive_cenn_experiment import run_symbolic_retrieval, run_diffuse_gap
out_dir=pathlib.Path('/content/ccm_cenn_feasibility' if pathlib.Path('/content').exists() else './ccm_cenn_feasibility')
out_dir.mkdir(exist_ok=True,parents=True)
for slots, ids, steps in [(8,16,400),(16,32,400),(32,64,400)]:
    report=run_symbolic_retrieval(steps=steps,slots=slots,ids=ids)
    (out_dir/f'learned_{slots}.json').write_text(json.dumps(report,indent=2))
    print('Cells',slots,'learned relative error:',report['history'][-1]['relative_error'])
gap=run_diffuse_gap()
(out_dir/'softmax_gap.json').write_text(json.dumps(gap,indent=2))
print('Diffuse attention approximation gaps (attention teacher only):')
for r in gap:
    if r['slots']==16 and r['temperature_multiplier'] in (1,4):
        print(r)

ModuleNotFoundError: No module named 'tinycenn_lm'

In [ ]:
#@title 4. Probe native first attention layer (GPU required; no student model trained)
if not torch.cuda.is_available():
    print('GPU unavailable: cannot claim real SmolLM2 attention-layer feasibility. Please choose a Colab GPU runtime.')
else:
    subprocess.check_call([sys.executable,'scripts/attention_concentration_smol360.py',
                           '--device','cuda','--lengths','128','256',
                           '--output',str(out_dir/'smol360_attention_diagnostic.json')])
    native=json.loads((out_dir/'smol360_attention_diagnostic.json').read_text())
    for x in native['results']:
        print('seq=',x['length'],'case=',x['text_case'],
              'top1mass=',round(x['topk']['1']['mean_probability_mass'],3),
              'top4mass=',round(x['topk']['4']['mean_probability_mass'],3),
              'top4 oracle MSE=',round(x['topk']['4']['output_relative_mse'],3))
    # A diagnostic criterion only, NOT an acceptance gate for deployment.
    go=all(x['topk']['4']['mean_probability_mass']>0.90 and
           x['topk']['4']['output_relative_mse']<0.05 for x in native['results'])
    print('Top-4 selective-cell feasibility screen:', 'PROMISING' if go else 'NOT PROVEN')

In [ ]:
#@title 5. Package reproducible results into a ZIP (even for negative outcomes)
import zipfile
archive=out_dir.parent/(out_dir.name+'_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out_dir.rglob('*')):
        if f.is_file():z.write(f,f.relative_to(out_dir))
print('Saved:',archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Non-Colab: ZIP is available at the printed path.')

## How to interpret the results

A successful symbolic-identifier lookup at 8–32 cells is not proof of continuous-key softmax approximation. The native 360M attention diagnostic is decisive for deciding whether top-k competitive cell routing is even *plausible*. If the first-layer heads have high entropy, a hard-winner cellular architecture loses important weighted mixtures: do not begin another expensive teacher-distillation run. Examine head-by-head dispersion and full-model held-out NLL in a separate study. No GPU performance claim is made here.